# Modelado

### 1. Enfoque elegido

Durante la fase de exploración de datos de este proyecto definimos tres preguntas que queríamos responder:

1. ¿Las empresas con mayor inversión en IA por empleado muestran mayores mejoras de productividad, controlando por industria y tamaño empresarial? Para abordarla, planteamos modelar `productivity_change_percent` usando `ai_investment_per_employee` e incorporar al menos `industry` y `company_size` como variables de control.

2.  ¿Las diferencias de nivel de adopción de IA entre industrias se mantienen una vez consideradas características como tamaño empresarial y país? Para abordarla, planteamos comparar industrias controlando por período, tamaño y contexto geográfico, en lugar de atribuir las diferencias únicamente al sector.

3. ¿Una mayor adopción de IA se relaciona con destrucción neta de empleo o con transformación de puestos de trabajo? Para responderla, planteamos construir un balance simple, por ejemplo `net_job_change = jobs_created - jobs_displaced`, y analizar su relación con `ai_adoption_rate`, `task_automation_rate`, industria y tamaño empresarial.

En primer lugar, debemos considerar que las tres variables target (`productivity_change_percent`, `ai_adoption_rate` y `net_job_change`) son contínuas. Esto nos orienta directamente hacia modelos de regresión. 

En segundo lugar, al tener una estructura de datos longitudinal, debemos modelarlo como un **panel**, donde para cada unidad muestral se tienen varias mediciones a lo largo del tiempo. 

Considerando estos puntos, a continuación planteamos un abordaje para cada pregunta:

**H1: Inversión en AI y productividad**

Target: `productivity_change_percent`

Modelo principal: Regresión lineal con efectos mixtos

Predictores:
- `ai_investment_per_employee`
- `industry`
- `company_size`
- `survey_year`
- `quarter`
- `ai_adoption_rate`

Si los predictores están correlacionados, utilizaríamos una regresión regularizada como Ridge o ElasticNet.

Como modelo no lineal para comparación, podríamos utilizar Random Forest o Gradient Boosting. Estos pueden detectar umbrales o efectos no lineales, pero no son tan adecuados como el modelo principal dado que nuestro objetivo es explicar la asociación. 

La fórmula del modelo sería: 
$$
productivity\_change = \beta_0+\beta_1 investment_{it}+\beta_2 industry_{i}+\beta_3 size_i + \beta_4 time_t +\beta_5 adoption_i + u_i + \epsilon_{it}
$$

Aquí, $u_i$ representa el efecto específico de cada compañía. Una regresión con efectos mixtos (GLMM) o una regresión de panel de efectos fijos (modelos TWFE) es preferible a una regresión OLS, dado que las mismas compañías aparecen varias veces. 


**H2: Adopción de IA por industria**

Target: `ai_adoption_rate`

Modelo principal: Regresión lineal con efectos mixtos
- Industria como la principal variable explicativa
- Controles para `company_size`, `country`, `survey_year` y `quarter`
- Efecto de la compañía para compensar las observaciones repetidas. 

Como modelo secundario también se puede ajustar un Gradient Boosting o Random Forest para identificar interacciones no lineales. Por ejemplo, si las diferencias entre industrias son más fuertes para compañías más grandes. 

La pregunta principal que buscamos responder aquí es si los coeficientes de industria se mantienen significativos luego de introducir los controles de tamaño, país y tiempo.


**H3: Transformación del empleo y AI**

En primer lugar, construiremos el target como 
```{python}
company_limpio["net_job_change"] = (
    company_limpio["jobs_created"]
    - company_limpio["jobs_displaced"]
)
```
En esta pregunta tenemos varios abordajes posibles:

**Abordaje de regresión:**

Usando `net_job_change` como el target, se puede aplicar:
- Regresión lineal si la distribución es razonablemente contínua.
- Random Forest o Gradient Boosting si se esperan comportamiento no lineales.
- Regresión Poisson o Binomial Negativa si deseamos modelar `jobs_created` y `jobs_displaced` de manera separada como conteos.

Los predictores más importantes serían:

- `ai_adoption_rate`
- `task_automation_rate`
- `ai_maturity_score`
- `industry`
- `company_size`
- `num_employees`
- `survey_year`
- `quarter`

**Abordaje de clasificación**

Creando una variable target categórica dicotómica
```{python}
company_limpio["employment_effect"] = np.select(
    [
        company_limpio["net_job_change"] > 0,
        company_limpio["net_job_change"] < 0
    ],
    [
        "net_creation",
        "net_displacement"
    ],
    default="neutral"
)
```
Se podría usar:
- Regresión logística multinomial
- Random Forest o Gradient Boosting para una clasificación no lineal
- Evaluación con macro-F1, balanced accuracy y matriz de confusión, dado que las categorías podrían no estar balanceadas. 

Por último, respecto del split train/test, consideramos que la mejor manera es realizar un splitting agrupado por company_id. De esa manera, todas las observaciones de una misma empresa estarán en uno de los conjuntos de datos.

### 2. Preparación final

### 3. Implementación y evaluación

### 4. Interpretación de resultados

### 5. Vuelta a las preguntas del P1

### 6. Conclusiones finales